[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Aggregation &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Print Oslo's mean, row 1, rounded to two places, and the date and time of its warmest hour.

In [2]:
oslo = readings[1]

print(oslo.mean().round(2))
print(START + timedelta(hours=int(oslo.argmax())))

6.5
2025-07-17 15:00:00


Oslo has no missing hours, so `mean` and `argmax` need no `nan` version.

**2.** Print the 25th, 50th and 75th percentiles of Tromso's readings, row 3.

In [3]:
print(np.percentile(readings[3], [25, 50, 75]))

[-2.4  3.5  9.4]


The middle half of Tromso's hours lie between the first and the last of these, and the distance between them is called the interquartile range.

**3.** Print Svalbard's standard deviation with `np.nanstd`, rounded to two places, and the date and time of its warmest hour with `np.nanargmax`.

In [4]:
svalbard = readings[2]

print(np.nanstd(svalbard).round(2))
print(START + timedelta(hours=int(np.nanargmax(svalbard))))

6.73
2025-07-13 15:00:00


`np.argmax` would have returned the first missing hour, as `np.argmin` did in the Common errors.

**4.** Find the coldest reading in the whole grid with `np.nanmin`. `np.nanargmin` on the grid gives a position in the flattened array, and `np.unravel_index(position, readings.shape)` turns it into a row and a column; print both, and the station's name.

In [5]:
position = np.nanargmin(readings)
row, column = np.unravel_index(position, readings.shape)

print(np.nanmin(readings), position)
print(row, column, list(STATIONS)[row])

-17.3 17787
2 267 Svalbard


The flattened position counts along the rows one after another, so 17,787 is two full rows of 8,760 and then 267 hours into the third.

**5.** Make Oslo's running count of hours below freezing with `np.cumsum` on a mask, and print the date and time of its 1,000th frost hour.

In [6]:
frost_count = np.cumsum(readings[1] < 0)

print(frost_count[-1])
print(START + timedelta(hours=int(np.argmax(frost_count >= 1000))))

1848
2025-02-26 00:00:00


`True` counts as 1, so the running sum of a mask counts the `True` values so far.

---

&#8592; **Back to:** [Aggregation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/10-aggregation.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
